# Exclusive Gene Analysis: CAT vs Ensembl

This notebook rebuilds the gene-exclusive analysis directly from raw per-assembly `gene_presence` outputs in `qc_metrics/`.

It is intended for the HPC results directory, where the current aggregate artifacts are incomplete or misleading for CAT-only vs Ensembl-only interpretation.

Questions this notebook targets:
- Is CAT's higher gene count coming from canonical GRCh38-like genes or mostly extra CAT-added models?
- What fraction of CAT-only genes are `MSTRG`, clone-style loci, `CHM13_G` IDs, or standard symbols?
- For CAT-only genes, how much comes from CAT source models versus Liftoff?
- Which protein-coding catalog-like genes are repeatedly CAT-only or Ensembl-only across assemblies?

In [ ]:
import pandas as pd
from exclusive_gene_analysis_lib import CATALOG_NAME_CLASSES, make_overview_figure, run_analysis

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)

results = run_analysis()
paths = results['paths']
files = results['files']

print('OUTPUT_DIR:', paths['output_dir'])
print('QC_DIR:', paths['qc_dir'])
print('CAT_CACHE_DIR:', paths['cat_cache_dir'])

print('gene_presence files:', len(files['gene_presence_files']))
print('Ensembl gene count files:', len(files['ensembl_gene_count_files']))
print('CAT gene count files:', len(files['cat_gene_count_files']))
print('analysis output:', paths['analysis_dir'])

OSError: [Errno 30] Read-only file system: '/hps'

In [ ]:
assembly = results['assembly_overview']
exclusive_obs = results['exclusive_obs']
exclusive_rollup = results['exclusive_rollup']
cat_only = results['cat_only_enriched']

summary = pd.DataFrame([
    {
        'median CAT - Ensembl total genes': assembly['delta_total'].median(),
        'median CAT - Ensembl non-fallback genes': assembly['delta_non_fallback'].median(),
        'median CAT - Ensembl catalog-like genes': assembly['delta_catalog_like'].median(),
        'median CAT - Ensembl protein_coding genes': assembly['delta_protein_coding'].median(),
        'median CAT - Ensembl protein_coding catalog-like genes': assembly['delta_protein_coding_catalog_like'].median(),
        'CAT-only exclusive observations': int((exclusive_obs['exclusive_side'] == 'cat_only').sum()),
        'Ensembl-only exclusive observations': int((exclusive_obs['exclusive_side'] == 'ensembl_only').sum()),
        'CAT-only exclusive genes': int((exclusive_rollup['n_assemblies_cat_only'] > 0).sum()),
        'Ensembl-only exclusive genes': int((exclusive_rollup['n_assemblies_ensembl_only'] > 0).sum()),
    }
])
display(summary.T.rename(columns={0: 'value'}))

display(results['exclusive_name_summary'])
display(results['exclusive_biotype_summary'].head(30))
display(results['cat_only_source_summary'].head(30))

In [ ]:
cat_pc_catalog = (
    cat_only[
        cat_only['cat_biotype'].eq('protein_coding')
        & cat_only['catalog_like']
    ]
    .groupby(['gene_name', 'cat_source'], dropna=False)
    .size()
    .reset_index(name='n_assemblies_cat_only')
    .sort_values(['n_assemblies_cat_only', 'gene_name'], ascending=[False, True])
)

ens_pc_catalog = (
    exclusive_rollup[
        exclusive_rollup['ensembl_biotype'].eq('protein_coding')
        & exclusive_rollup['catalog_like']
        & (exclusive_rollup['n_assemblies_ensembl_only'] > 0)
    ][['gene_name', 'n_assemblies_ensembl_only', 'ensembl_chrom']]
    .sort_values(['n_assemblies_ensembl_only', 'gene_name'], ascending=[False, True])
)

display(cat_pc_catalog.head(50))
display(ens_pc_catalog.head(50))

cat_name_breakdown = (
    cat_only[
        cat_only['cat_biotype'].eq('protein_coding')
        & (cat_only['cat_source'] == 'CAT')
    ]
    .groupby('name_class', dropna=False)
    .size()
    .reset_index(name='n_observations')
    .sort_values('n_observations', ascending=False)
)
display(cat_name_breakdown)

In [ ]:
fig, axes = make_overview_figure(results, top_n=20)
fig